--- Day 11: Plutonian Pebbles ---
The ancient civilization on Pluto was known for its ability to manipulate spacetime, and while The Historians explore their infinite corridors, you've noticed a strange set of physics-defying stones.

At first glance, they seem like normal stones: they're arranged in a perfectly straight line, and each stone has a number engraved on it.

The strange part is that every time you blink, the stones change.

Sometimes, the number engraved on a stone changes. Other times, a stone might split in two, causing all the other stones to shift over a bit to make room in their perfectly straight line.

As you observe them for a while, you find that the stones have a consistent behavior. Every time you blink, the stones each simultaneously change according to the first applicable rule in this list:

If the stone is engraved with the number 0, it is replaced by a stone engraved with the number 1.
If the stone is engraved with a number that has an even number of digits, it is replaced by two stones. The left half of the digits are engraved on the new left stone, and the right half of the digits are engraved on the new right stone. (The new numbers don't keep extra leading zeroes: 1000 would become stones 10 and 0.)
If none of the other rules apply, the stone is replaced by a new stone; the old stone's number multiplied by 2024 is engraved on the new stone.
No matter how the stones change, their order is preserved, and they stay on their perfectly straight line.

How will the stones evolve if you keep blinking at them? You take a note of the number engraved on each stone in the line (your puzzle input).

If you have an arrangement of five stones engraved with the numbers 0 1 10 99 999 and you blink once, the stones transform as follows:

The first stone, 0, becomes a stone marked 1.
The second stone, 1, is multiplied by 2024 to become 2024.
The third stone, 10, is split into a stone marked 1 followed by a stone marked 0.
The fourth stone, 99, is split into two stones marked 9.
The fifth stone, 999, is replaced by a stone marked 2021976.
So, after blinking once, your five stones would become an arrangement of seven stones engraved with the numbers 1 2024 1 0 9 9 2021976.

Here is a longer example:

Initial arrangement:
125 17

After 1 blink:
253000 1 7

After 2 blinks:
253 0 2024 14168

After 3 blinks:
512072 1 20 24 28676032

After 4 blinks:
512 72 2024 2 0 2 4 2867 6032

After 5 blinks:
1036288 7 2 20 24 4048 1 4048 8096 28 67 60 32

After 6 blinks:
2097446912 14168 4048 2 0 2 4 40 48 2024 40 48 80 96 2 8 6 7 6 0 3 2
In this example, after blinking six times, you would have 22 stones. After blinking 25 times, you would have 55312 stones!

Consider the arrangement of stones in front of you. How many stones will you have after blinking 25 times?

In [2]:
class Stones:
    def __init__(self, value, next=None):
        self.value = value
        self.next = next


    def blink(self):
        next = self
        while next:
            next = next.blink_next()

    def count_stones(self):
        count = 0
        next = self
        while next:
            count += 1
            next = next.next
        return count

    def blink_next(self):
        # print(f'{self.value}, ', end = '')
        if self.value == '0':
            self.value = '1'
            return self.next
        elif len(self.value) % 2 == 0:
            self.split()
            return self.next.next
        else:
            self.value = str(int(self.value) * 2024)
            return self.next

    def split(self):
        v1, v2 = Stones.half(self.value)
        self.value = v1
        self.next = Stones(v2, self.next)

    @staticmethod
    def half(string):
        splitpoint = len(string) // 2
        halves = [s.lstrip('0') for s in (string[:splitpoint], string[splitpoint:])]
        halves = ['0' if h == '' else h for h in halves]
        return halves

In [3]:
# load initials:
input = "27 10647 103 9 0 5524 4594227 902936"
for i, num in enumerate(input.split()):
    if i == 0:
        first_stone = prev = Stones(num)
    else:
        if num == "902936":
            pass
        prev.next = Stones(num)
        prev = prev.next

In [4]:
for _ in range(25):
    first_stone.blink()
first_stone.count_stones()

229043

Your puzzle answer was 229043.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
The Historians sure are taking a long time. To be fair, the infinite corridors are very large.

How many stones would you have after blinking a total of 75 times?

In [15]:
# load initials:
input = "27 10647 103 9 0 5524 4594227 902936"
for i, num in enumerate(input.split()):
    if i == 0:
        first_stone = prev = Stones(num)
    else:
        if num == "902936":
            pass
        prev.next = Stones(num)
        prev = prev.next

num_stones = {}
for i in range(30):
    first_stone.blink()
    num_stones[i] = first_stone.count_stones()
num_stones

{0: 11,
 1: 16,
 2: 21,
 3: 39,
 4: 54,
 5: 82,
 6: 130,
 7: 189,
 8: 279,
 9: 432,
 10: 676,
 11: 997,
 12: 1505,
 13: 2305,
 14: 3563,
 15: 5267,
 16: 8034,
 17: 12385,
 18: 18547,
 19: 28183,
 20: 42834,
 21: 65536,
 22: 99262,
 23: 149589,
 24: 229043,
 25: 347592,
 26: 526000,
 27: 801345,
 28: 1215691,
 29: 1847873}

In [ ]:
f'{(1847873 / 1215691) ** 75:e}'

'4.351799e+13'

In [14]:
import psutil
import os

process = psutil.Process(os.getpid())
# Get memory usage in MB
ram_used_mb = process.memory_info().rss / (1024 * 1024)
print(f"Current RAM usage: {ram_used_mb:.2f} MB")

Current RAM usage: 345.94 MB


It takes prohibitively long. Ideas for improvement:
- parallel computing
- precompute chunks. Eg we could populate a table where for every number, we calculate what we get after 10 iterations. 

Still, the combinatorial explosion seems overwhelming. How could we make hashing the calculations really fast?
I am thinking of something like this:
We do one calculation. We store the result of that calculation in a dict. The next time we do that calculation, we just look up the result. What's more, when we do one step further in the calculation, we can store *that* result too. 

Now we could store every result for every calculation for looong chains in that database. Problem: Quickly needs a lot of storage. The other option would be to store the whole thing as a tree: You look up the next step, then you look up the answer to that... Problem: you quickly need an enormous amount of lookups

I suspect there is a way to balance these. How many results further you store and for which numbers (more for numbers that get used more often). The other thing is the sheer number of objects to calculate. It is around 100 TB just for the numbers! Not happening. But we don't need the whole thing, since we just need some intermediate values and need to know what they will blow up to. 


Claude let me know that...the position of the stones doesn't actually matter to the puzzle. Duh! I had a lot of unnecessary overhead because of this. Ok let's rewrite.

In [32]:
from collections import defaultdict

# Initialize:
start_stones = defaultdict(int)
for num in input.split():
    num = int(num)
    start_stones[num] += 1

# Main loop:
old_stones = start_stones
for _ in range(75):
    new_stones = defaultdict(int)
    for num, count in old_stones.items():
        if num == 0:
            new_stones[1] += count
        elif len(s:=str(num)) % 2 == 0:
            mid = len(s) // 2
            h1, h2 = (int(h) for h in (s[:mid], s[mid:]))
            new_stones[h1] += count
            new_stones[h2] += count
        else:
            new_stones[num * 2024] += count
    old_stones = new_stones

# Count stones:
sum(new_stones.values()), len(new_stones)

(272673043446478, 3735)

That's the right answer! You are one gold star closer to finding the Chief Historian.

You have completed Day 11! You can [Share] this victory or [Return to Your Advent Calendar].